Pixelization: Many Visibilities Preparation
===========================================

To perform many visibility modeling, a matrix called `nufft_precision_operator` is created and used, which encodes information
and symmetries into the Fourier transform operation performed when modeling interferometer datasets, in a way that
exploits the sparsity of the inversion — pixelizations, linear light profiles, or both — and means a very small
amount of memory or VRAM is used.

The details can be found in the source code, but you do not need to know them to do science with the code,
nevertheless this ultimately means datasets exceeding millions of visibilities can be modeled in under an hour on a GPU.

The time to compute this matrix can vary between seconds and hours, depending on the number of visibilities in the
dataset, the number of image pixels in the real space mask and if a CPU or GPU is used. If this matrix is not saved
and loaded from hard disk, this recalculation would need to be performed before every model-fit, which if for your
setup and hardware takes hours would be prohibitive.

On HPC GPUs via JAX, this computation is fast even for large datasets with many visibilities, with profiling
of high resolution datasets with over 1 million visibilities showing that computation takes under 20 seconds. For
10s or 100s of millions of visibilities computation on a GPU may stretch to minutes, but this is still very fast.
If you are lucky enough to have a modern enough GPU, you can therefore compute this matrix on-the-fly during modeling.

On consumer laptop GPUs or CPU, for datasets with over 100000 visibilities and many pixels in their real-space mask, this
computation may take 10 minutes or hours (for the small dataset loaded above its miliseconds). Computing it once,
in this script, saving it to hard-disk, and loading it for modeling is therefore recommended.

On CPU, the `show_progress` input outputs  a progress bar to the terminal so you can monitor the computation,
which is useful when it is slow.

This example therefore creates the `nufft_precision_operator` matrix using independent Python code and saves it to hard-disk
for modeling. The `cpu_fast_modeling` example loads this matrix from hard-disk if it is available,
and computes it from scratch if not.

__Contents__

- **High Resolution Dataset:** A high-resolution `uv_wavelengths` file for ALMA is available in a separate repository that hosts.
- **Profiling Dataset:** The code above loads a dataset with very few visibilities and a low resolution real space mask, so.
- **Curvature Preload:** Pixelized source modeling requires dense linear algebra operations.
- **Curvature Preload Output:** We now output the `nufft_precision_operator` object to hard-disk, so it can be loaded quickly in.
- **Wrap Up:** Summary of the script and next steps.

__High Resolution Dataset__

A high-resolution `uv_wavelengths` file for ALMA is available in a separate repository that hosts large files which
are too big to include in the main `autolens_workspace` repository:

https://github.com/PyAutoLabs/autolens_workspace_large_files

After downloading the file, place it in the directory:

`autolens_workspace/dataset/interferometer/alma`

You can then compute the `nufft_precision_operator` matrix for this dataset by uncommenting
the line `dataset_name = "alma"` below.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path
import subprocess
import sys
import time

import autolens as al
import autolens.plot as aplt

__Dataset + Masking__ 

Load the `Interferometer` data, define the visibility and real-space masks.

In [ ]:
mask_radius = 3.0

real_space_mask = al.Mask2D.circular(
    shape_native=(256, 256), pixel_scales=0.1, radius=mask_radius
)

dataset_name = "simple"
# dataset_name = "alma"
dataset_path = Path("dataset") / "interferometer" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    subprocess.run(
        [sys.executable, "scripts/interferometer/simulator.py"],
        check=True,
    )

dataset = al.Interferometer.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    uv_wavelengths_path=dataset_path / "uv_wavelengths.fits",
    real_space_mask=real_space_mask,
    transformer_class=al.TransformerNUFFT,
)

aplt.subplot_interferometer_dirty_images(dataset=dataset)

__Profiling Dataset__

The code above loads a dataset with very few visibilities and a low resolution real space mask, so the 
`nufft_precision_operator` computation is fast.

Real datasets often have 100,000+ visibilities, and a high resolution real space mask, which makes the 
`nufft_precision_operator` computation much slower.

It may therefore be useful to profile the run times for different dataset sizes using the code below, which overwrites 
the dataset above. This will allow you to plan ahead how long the `nufft_precision_operator` computation will take for your 
dataset, and whether doing it on a HPC is necessary.

This code is commented out by default, so your dataset is used instead, but you can uncomment it to run the profiling.

In [ ]:
# ### Key run time parameters ###
#
# mask_radius = 3.0
# total_visibilities = 1000000
#
# ### Setup Data ###
#
# real_space_mask = al.Mask2D.circular(
#     shape_native=(800, 800), pixel_scales=0.05, radius=mask_radius
# )
#
# data = al.Visibilities(np.random.normal(loc=0.0, scale=1.0, size=total_visibilities) + 1j * np.random.normal(
#     loc=0.0, scale=1.0, size=total_visibilities
# ))
#
# noise_map = al.VisibilitiesNoiseMap(np.ones(total_visibilities) + 1j * np.ones(total_visibilities))
#
# uv_wavelengths = np.random.uniform(
#     low=-300.0, high=300.0, size=(total_visibilities, 2)
# )
#
# dataset = al.Interferometer(
#     data=data,
#     noise_map=noise_map,
#     uv_wavelengths=uv_wavelengths,
#     real_space_mask=real_space_mask,
#     transformer_class=al.TransformerNUFFT,
# )

__Curvature Preload__

Pixelized source modeling requires dense linear algebra operations. These calculations are greatly accelerated
using an alternative mathematical approach called the **sparse linear algebra formalism**.

You do not need to understand the full details of the method, but the key point is:

- It exploits the **sparsity** of the matrices used in pixelized source reconstruction.
- This leads to a **significant speed-up on GPU or CPU**, using JAX to perform the linear algebra calculations.

To enable this feature, we call `apply_sparse_operator()` on the dataset. This computes and stores a
preload matrix, which reused in all subsequent pixelized source fits.

As discussed above, the computation of this matrix can take a long time for datasets with many visibilities
and high resolution real-space masks, unless a modern GPU is used.

We comment out the sparse linear algebra calculation below as we are going to illustrate how you can compute it on CPU.

The code has the following inputs:

- `chunk_k`: The chunk size of visibilities to process at a time. Decreasing this value decreases the memory
  requirements of the computation, but increases the run time. You should set this as high as your system's
  memory allows.
  
- `show_progress`: Whether to output a progress bar to the terminal, which is on here as for runs which take over
  an hour this is useful to monitor.

- `show_memory`: Whether to output memory usage to the terminal, which is useful to ensure your system has enough
  memory to complete the computation.

In [ ]:
dataset = dataset.apply_sparse_operator(
    use_jax=True,
    chunk_k=2048,
    show_progress=True,
    show_memory=True,
)

__Curvature Preload Output__

We now output the `nufft_precision_operator` object to hard-disk, so it can be loaded quickly in the 
`cpu_fast_modeling` example.

We save it using a numpy `npy` file, which compresses the data to save hard-disk space, and put it in the 
dataset folder so it can be easily found. 

In [ ]:
nufft_precision_operator = dataset.psf_precision_operator_from(
    use_jax=True,
    chunk_k=2048,
    show_progress=True,
    show_memory=True,
)

np.save(
    file=dataset_path / f"nufft_precision_operator_{mask_radius}.npy",
    arr=nufft_precision_operator,
    allow_pickle=False,
)

To load the `nufft_precision_operator` matrix from hard-disk in your model-fit, you can use the code:

In [ ]:
nufft_precision_operator = np.load(
    file=dataset_path / f"nufft_precision_operator_{mask_radius}.npy",
    allow_pickle=False,
)

__Wrap Up__

This example has demonstrated how to set up the linear algebra to perform fast pixelized source modeling on
interferometer datasets with many visibilities.